In [4]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "packages/research").is_dir() and (candidate / "packages/pipeline").is_dir():
            return candidate
    raise RuntimeError("Could not find IL_ideation repo root.")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
from openai import OpenAI
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

from agentlightning import LitAgent, Rollout, Trainer, VERL, reward
from packages.research.agent_loops import AgentLoopConfig, list_agent_loops, run_agent_loop
from packages.research.local_chat_models import make_chat_model, make_structured_llm

MODEL_ID = os.environ.get("QWEN_MODEL_ID", "Qwen/Qwen3-1.7B")
QWEN_OPENAI_BASE_URL = os.environ.get("RESEARCH_LLM_BASE_URL", "http://127.0.0.1:8001/v1")
QWEN_API_KEY = os.environ.get("RESEARCH_LLM_API_KEY", "EMPTY")


def make_langchain_qwen_model(
    *,
    model_id: str = MODEL_ID,
    base_url: str = QWEN_OPENAI_BASE_URL,
    api_key: str = QWEN_API_KEY,
    temperature: float = 0.2,
    max_tokens: int = 512,
    timeout: int = 120,
) -> ChatOpenAI:
    """LangChain-compatible Qwen chat model for agent-loop style use."""
    return ChatOpenAI(
        model=model_id,
        api_key=api_key,
        base_url=base_url,
        temperature=temperature,
        max_tokens=max_tokens,
        timeout=timeout,
    )

model = make_langchain_qwen_model()


In [5]:
# Qwen-backed rule builder running through the real grammar loop
from packages.pipeline import grammar_graph as grammar_tools
from packages.research.agent_loops.grammar_loop import (
    build_structural_rules,
    make_structured_agent_from_chat_model,
)

# grammar_loop will inject the live vocabulary and sampled few-shot structural rules.
qwen_rule_builder_agent = make_structured_agent_from_chat_model(
    model=model,
    schema=grammar_tools.StructuralRules,
    role="rule_builder_agent",
)

grammar_state = build_structural_rules(
    "Design a small quadruped robot for stair climbing.",
    rule_builder_agent=qwen_rule_builder_agent,
    population=2,
    max_attempts=3,
)


In [ ]:
print(grammar_state)

{'structural_rules': {'HIP_STACK': ['hip_yaw', 'hip_roll', 'hip_pitch'], 'LEG': ['HIP_STACK', 'thigh_link', 'knee_joint', 'shank_link', 'foot_pad'], 'S': ['BODY'], 'BODY_CHAIN': ['BODY_SEG'], 'TERMINAL_CONTACT': ['foot_pad'], 'DIGITIGRADE_LEG': ['hip_pitch', 'thigh_link', 'reverse_knee', 'heel_link', 'toe_pad'], 'SPRING_LEG': ['hip_joint', 'spring_thigh', 'compliant_knee', 'spring_shank', 'wide_foot'], 'SIDEWINDER_CHAIN': ['YAW_SEG', 'PITCH_SEG', 'SIDEWINDER_CHAIN'], 'PERCHING_LEG': ['hip_joint', 'shin_link', 'ankle_joint', 'opposable_talon'], 'TRUNK': ['SPRAWLED_LEG_PAIR', 'SPRAWLED_LEG_PAIR'], 'TAIL': ['tail_yaw', 'tail_link', 'tail_tip'], 'ADHESIVE_FOOT': ['toe_splay_joint', 'adhesive_pad'], 'SPINE_CHAIN': ['BODY_SEG', 'spine_yaw', 'SPINE_CHAIN'], 'SEG_CHAIN': ['SEG', 'flex_joint', 'SEG_CHAIN'], 'SEG': ['body_segment', 'LEG_PAIR']}}


In [ ]:
from packages.research.agent_loops.grammar_loop import GRAMMAR_LOOP_TOOLS
from packages.pipeline.grammar_graph import RoboGraphState
from packages.research.agent_loops.grammar_loop import STRUCTURAL_RULE_GRAPH_NODES, STRUCTURAL_RULE_GRAPH_NODE_ORDER,StructuralRuleLoopContext

agent_prompt_keys = [key for key in GRAMMAR_LOOP_TOOLS.keys() if "prompt" in key]
agent_prompts = [GRAMMAR_LOOP_TOOLS[key] for key in agent_prompt_keys]
tools = GRAMMAR_LOOP_TOOLS.keys() - agent_prompt_keys
print(tools)

{'repair_structural_rule_node_names', 'fetch_grammar_from_db', 'repair_structural_rule_node_names_safely', 'sample_rule_builder_examples', 'compile_structural_rules'}


In [ ]:
from typing import Any
from langgraph.graph import END, START, StateGraph
from packages.pipeline.observability import (
    get_langsmith_trace_id,
    langsmith_observation,
    langsmith_trace_context,
)
from packages.research.agent_loops.grammar_loop import (
    _bind_structural_rule_node,
    _compile_result_prompt_payload,
    build_initial_structural_rule_state,
    make_structural_rule_loop_context,
    route_after_evaluation,
    route_after_normalize,
    summarize_hitl_state,
)


def _route_after_compiling(state: RoboGraphState) -> str:
    compile_result = state.get("compile_result")
    attempts = int(state.get("attempts") or 0)
    max_attempts = int(state.get("max_attempts") or 1)
    compile_payload = (
        _compile_result_prompt_payload(compile_result)
        if compile_result is not None
        else {"compile_safe": False, "compile_error": "missing compile_result"}
    )
    if compile_result and bool(compile_result):
        decision = "continue"
    elif attempts < max_attempts:
        decision = "rebuild"
    else:
        decision = "continue"
    with langsmith_observation(
        "route_after_compiling",
        as_type="chain",
        input={
            "attempts": attempts,
            "max_attempts": max_attempts,
            "compile_result": compile_payload,
        },
        metadata={"router_decision": decision},
    ) as observation:
        observation.update(output={"decision": decision})
    return decision


def _route_after_evaluation_traced(state: RoboGraphState, context: StructuralRuleLoopContext) -> str:
    decision = route_after_evaluation(state, context)
    with langsmith_observation(
        "route_after_evaluation",
        as_type="chain",
        input={
            "attempts": int(state.get("attempts") or 0),
            "max_attempts": int(state.get("max_attempts") or 1),
            "checklist": state.get("checklist"),
        },
        metadata={"router_decision": decision},
    ) as observation:
        observation.update(output={"decision": decision})
    return decision


def _compiler_centric_loop(*, context: StructuralRuleLoopContext) -> Any:
    graph = StateGraph(RoboGraphState)
    for node_name in STRUCTURAL_RULE_GRAPH_NODE_ORDER:
        graph.add_node(
            node_name,
            _bind_structural_rule_node(STRUCTURAL_RULE_GRAPH_NODES[node_name], context),
        )

    graph.add_edge(START, "normalize_query")
    graph.add_conditional_edges(
        "normalize_query",
        lambda state: route_after_normalize(state, context),
        {
            "await_human_confirmation": "await_human_confirmation",
            "make_initial_checklist": "make_initial_checklist",
        },
    )
    graph.add_edge("await_human_confirmation", "summarize_hitl")
    graph.add_edge("make_initial_checklist", "build_structural_rules")
    graph.add_edge("build_structural_rules", "resolve_grammar_node_names")
    graph.add_edge("resolve_grammar_node_names", "compile_structural_rules")
    graph.add_conditional_edges(
        "compile_structural_rules",
        _route_after_compiling,
        {
            "rebuild": "build_structural_rules",
            "continue": "evaluate_rules",
        },
    )
    graph.add_conditional_edges(
        "evaluate_rules",
        lambda state: _route_after_evaluation_traced(state, context),
        {
            "build_structural_rules": "build_structural_rules",
            "summarize_hitl": "summarize_hitl",
        },
    )
    graph.add_edge("summarize_hitl", END)
    return graph.compile()


def run_compiler_centric_loop(
    prompt: str,
    *,
    initial_state: RoboGraphState | None = None,
    context: StructuralRuleLoopContext | None = None,
    population: int | None = None,
    max_attempts: int = 2,
) -> RoboGraphState:
    state = build_initial_structural_rule_state(
        prompt,
        initial_state,
        population=population,
        max_attempts=max_attempts,
    )
    loop_context = context or make_structural_rule_loop_context()
    graph = _compiler_centric_loop(context=loop_context)
    with langsmith_observation(
        "robogrammar.compiler_centric_loop",
        as_type="agent",
        input={"prompt": prompt, "population": state.get("population")},
        metadata={"max_attempts": state.get("max_attempts")},
    ) as observation:
        with langsmith_trace_context():
            final_state = graph.invoke(state, config={"callbacks": []})
        trace_id = get_langsmith_trace_id()
        if trace_id:
            final_state["langsmith_trace_id"] = trace_id
        final_state["hitl"] = summarize_hitl_state(final_state)
        observation.update(
            output=final_state["hitl"],
            metadata={
                "attempts": int(final_state.get("attempts") or 0),
                **_compile_result_prompt_payload(final_state.get("compile_result")),
            },
        )
        return final_state



In [ ]:
# Root cause: compile-invalid retries need exact compiler feedback in the next
# rule-builder prompt. The compiler-centric loop rebuilds immediately after a
# failed compile, so this variant injects a compiler-specific checklist item
# before routing back to build_structural_rules.


def _compiler_feedback_node(state: RoboGraphState, _context: StructuralRuleLoopContext) -> dict[str, Any]:
    compile_result = state.get("compile_result")
    compile_payload = _compile_result_prompt_payload(compile_result)
    critiques: list[str] = []
    if compile_payload["compile_error"]:
        error = compile_payload["compile_error"]
        critiques.append(f"Compiler error: {error.get('error_type')}: {error.get('error_message')}")
    if compile_payload["invalid_grammar_nodes"]:
        critiques.append(
            "Invalid GrammarNodes IDs: " + ", ".join(compile_payload["invalid_grammar_nodes"])
        )
    if compile_payload["unreachable_grammar_nodes"]:
        critiques.append(
            "Valid but unreachable from S: " + ", ".join(compile_payload["unreachable_grammar_nodes"])
        )
    checklist = state.get("checklist") or {"criteria": []}
    criteria = [
        item
        for item in checklist.get("criteria", [])
        if item.get("description") != "Compiler feedback is resolved."
    ]
    criteria.append(
        {
            "description": "Compiler feedback is resolved.",
            "isSuccessful": bool(compile_result),
            "critiques": critiques,
        }
    )
    with langsmith_observation(
        "compiler_feedback",
        as_type="chain",
        input=compile_payload,
        metadata={"compile_safe": bool(compile_result)},
    ) as observation:
        observation.update(output={"criteria": criteria[-1]})
    return {
        "checklist": {"criteria": criteria},
        "messages": ["compiler_feedback: compile diagnostics added to checklist."],
    }


def _to_xml(state: RoboGraphState) -> str:
    return ""

def _compiler_feedback_loop(*, context: StructuralRuleLoopContext) -> Any:
    graph = StateGraph(RoboGraphState)
    for node_name in STRUCTURAL_RULE_GRAPH_NODE_ORDER:
        graph.add_node(
            node_name,
            _bind_structural_rule_node(STRUCTURAL_RULE_GRAPH_NODES[node_name], context),
        )
    graph.add_node(
        "compiler_feedback",
        lambda state: _compiler_feedback_node(state, context),
    )

    graph.add_edge(START, "normalize_query")
    graph.add_conditional_edges(
        "normalize_query",
        lambda state: route_after_normalize(state, context),
        {
            "await_human_confirmation": "await_human_confirmation",
            "make_initial_checklist": "make_initial_checklist",
        },
    )
    graph.add_edge("await_human_confirmation", "summarize_hitl")
    graph.add_edge("make_initial_checklist", "build_structural_rules")
    graph.add_edge("build_structural_rules", "resolve_grammar_node_names")
    graph.add_edge("resolve_grammar_node_names", "compile_structural_rules")
    graph.add_edge("compile_structural_rules", "compiler_feedback")
    graph.add_conditional_edges(
        "compiler_feedback",
        _route_after_compiling,
        {
            "rebuild": "build_structural_rules",
            "continue": "evaluate_rules",
        },
    )
    graph.add_conditional_edges(
        "evaluate_rules",
        lambda state: _route_after_evaluation_traced(state, context),
        {
            "build_structural_rules": "build_structural_rules",
            "summarize_hitl": "summarize_hitl",
        },
    )
    graph.add_edge("summarize_hitl", END)
    return graph.compile()


def run_compiler_feedback_loop(
    prompt: str,
    *,
    initial_state: RoboGraphState | None = None,
    context: StructuralRuleLoopContext | None = None,
    population: int | None = None,
    max_attempts: int = 2,
) -> RoboGraphState:
    state = build_initial_structural_rule_state(
        prompt,
        initial_state,
        population=population,
        max_attempts=max_attempts,
    )
    loop_context = context or make_structural_rule_loop_context()
    graph = _compiler_feedback_loop(context=loop_context)
    with langsmith_observation(
        "robogrammar.compiler_feedback_loop",
        as_type="agent",
        input={"prompt": prompt, "population": state.get("population")},
        metadata={"max_attempts": state.get("max_attempts")},
    ) as observation:
        with langsmith_trace_context():
            final_state = graph.invoke(state, config={"callbacks": []})
        trace_id = get_langsmith_trace_id()
        if trace_id:
            final_state["langsmith_trace_id"] = trace_id
        final_state["hitl"] = summarize_hitl_state(final_state)
        observation.update(
            output=final_state["hitl"],
            metadata={
                "attempts": int(final_state.get("attempts") or 0),
                **_compile_result_prompt_payload(final_state.get("compile_result")),
            },
        )
        return final_state


In [ ]:
from __future__ import annotations

from typing import Any

import agentlightning as agl
from openai import OpenAI

TRAIN_DATA = [
    {
        "prompt": "Design a quadruped robot that can climb stairs.",
        "must_include": ["four legs", "hip joints", "knee joints"],
    },
    {
        "prompt": "Design a compact wheeled robot for indoor inspection.",
        "must_include": ["wheels", "sensor mount"],
    },
]


# rewards I want
1. "if task asks for stability" -> check ujoco pertrubation
2. 

SyntaxError: invalid syntax (905423522.py, line 21)